# IDX Exchange DS Internship - Week Four -  Baseline Model

* Train a Linear Regression as the first model.
* Evaluate using R² on the test set.
* Record baseline results.

--------

In [131]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import TargetEncoder

from sklearn.compose import TransformedTargetRegressor

This notebook will test various linear regression models. The first model will use the dataset created in week three, which will establish a baseline threshold that future models should exceed. Other linear regression models will be tested here to observe how changes to the dataset affect model performance. Furthermore, this notebook also contains the linear regression results using the feature engineered dataset created in week six. All models will be evaluated using $R^2$, MAPE, MdAPE, and MAE metrics.

## Import Data

In [132]:
# Read in the training and test sets created last week

train_and_val = pd.read_csv("week_3_train_2.csv")
testing = pd.read_csv("week_3_test_2.csv")

In [133]:
# Create a validation set out of the training data

# Training Set: January 2024 - February 2026
# Validation Set: March 2026
# Test Set: April 2026

training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265326, 47) (255033, 47) (10293, 47)


In [134]:
training.columns

Index(['ViewYN', 'PoolPrivateYN', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListOfficeName', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'PropertyMonth', 'ClosePriceLog', 'LivingAreaLog',
       'LotSizeSquareFeetLog', 'AssociationFeeLog', 'Bamboo', 'Brick',
       'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile',
       'Unknown', 'Vinyl', 'Wood', 'ListOfficeName_enc', 'CountyOrParish_enc',
       'HighSchoolDistrict_enc', 'City_enc', 'PostalCode_enc',
       'YearBuilt_enc'],
      dtype='object')

In [135]:
def median_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true))

def metrics(y_true, y_pred, logged=True):
    # Note: you have to undo the log transform on ClosePrice by using np.exp()
    if logged == True:
        y_true = np.exp(y_true)
        y_pred = np.exp(y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    mdape = median_absolute_percentage_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, mape * 100, mdape * 100, mae

def metric_df(r2_train, mape_train, mdape_train, mae_train,
              r2_val, mape_val, mdape_val, mae_val,
              r2_test, mape_test, mdape_test, mae_test):
    data = {"r2": [r2_train, r2_val, r2_test],
            "MAPE": [mape_train, mape_val, mape_test],
            "MdAPE": [mdape_train, mdape_val, mdape_test],
            "MAE": [f"${round(mae_train, 2)}", f"${round(mae_val, 2)}", f"${round(mae_test, 2)}"]}
    df = pd.DataFrame(data)
    df.index = ["Train Set", "Validation Set", "Test Set"]
    return df

In [136]:
# Columns not being used: "ListingKey", "ListingId", "ClosePrice", "ListOfficeName", "CountyOrParish", "YearBuilt",
#         "City", "HighSchoolDistrict", "PostalCode", "LivingArea", "LotSizeSquareFeet", "AssociationFee"

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "PropertyMonth", "ClosePriceLog"]

# The x_train window will be set per model, so don't define it here

x_val = validation[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc"]]  
y_val = validation["ClosePriceLog"]


x_test = testing[["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc"]]
y_test = testing["ClosePriceLog"]

## Model One: Baseline

Train a baseline model on the data created in week three. Note that the target variable for this model is ClosePriceLog rather than ClosePrice. This is done to account for the major right-skew in ClosePrice. However, it means that the model actually predicts the log of ClosePrice. When evaluating the model, the predictions and target variable have $e$ applied to them, undoing this log transform and returning the data to its original form.

### Train Set: January 2024 - February 2026

In [137]:
# Use the full set of available months (26 months)

x_train = training[common_cols].drop(columns = ["PropertyMonth", "ClosePriceLog"])
y_train = training["ClosePriceLog"]

In [138]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [139]:
model_one_v1 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v1

,r2,MAPE,MdAPE,MAE
Train Set,0.834530,13.624931,10.555025,$153174.83
Validation Set,0.823092,14.035858,10.948419,$159771.79
Test Set,0.845539,13.254369,10.399052,$155596.03


This is a very promising baseline! The model seems to perform well using all four error metrics. However, to make sure that the model is not overfitting on a subset of months, test other training windows and verify that they perform similarly.

### Train Set: February 2025 - February 2026

In [140]:
# Shorten the training window to the year preceding the validation set (12 months)

past_limit = pd.to_datetime("2025-01-01")
present_limit = pd.to_datetime("2026-02-01")

x_train_with_month = training[common_cols]
x_train_with_month.loc[:, "PropertyMonth"] = pd.to_datetime(x_train_with_month["PropertyMonth"])
x_train = x_train_with_month[(x_train_with_month["PropertyMonth"] >= past_limit) & (x_train_with_month["PropertyMonth"] <= present_limit)]
y_train = x_train["ClosePriceLog"]
x_train = x_train.drop(columns = ["PropertyMonth", "ClosePriceLog"])

In [141]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [142]:
model_one_v2 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v2

,r2,MAPE,MdAPE,MAE
Train Set,0.828092,13.827305,10.729606,$154534.72
Validation Set,0.822924,14.052443,10.980927,$159940.96
Test Set,0.844659,13.300858,10.388599,$156199.34


The model continues to perform well when the training window is cut down to one year. This indicates that the model is not overfitting to a specific time, and is instead learning robust predictive features. But we should try one more training window to be thorough.

### Train Set: July 2024 - February 2026

In [143]:
# Set the training window to 20 months preceding the validation set

past_limit = pd.to_datetime("2024-06-01")
present_limit = pd.to_datetime("2026-02-01")

x_train_with_month = training[common_cols]
x_train_with_month.loc[:, "PropertyMonth"] = pd.to_datetime(x_train_with_month["PropertyMonth"])
x_train = x_train_with_month[(x_train_with_month["PropertyMonth"] >= past_limit) & (x_train_with_month["PropertyMonth"] <= present_limit)]
y_train = x_train["ClosePriceLog"]
x_train = x_train.drop(columns = ["PropertyMonth", "ClosePriceLog"])

In [144]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [145]:
model_one_v3 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_one_v3

,r2,MAPE,MdAPE,MAE
Train Set,0.833710,13.620486,10.558315,$153167.12
Validation Set,0.823034,14.055172,10.963304,$159862.07
Test Set,0.845240,13.284478,10.398347,$155840.17


The model holds steady in the same ranges as before, despite the change in training window. This suggests that the model is robust to changes in time.

## Model Two: No Log Transform on Target

In order to test the validity of the log transform on ClosePrice, we will test a control model that does not perform this log transform. During the preprocessing stage, high-cardinality categorical variables were target encoded on ClosePriceLog. To test this model properly, target encoding must be re-computed on ClosePrice.

In [146]:
# Redo the target encoding on the categorical variables, but using ClosePrice instead of ClosePriceLog

to_encode_train = train_and_val[["ListOfficeName", "CountyOrParish", "HighSchoolDistrict", "City", "PostalCode", "YearBuilt"]]
to_encode_test = testing[["ListOfficeName", "CountyOrParish", "HighSchoolDistrict", "City", "PostalCode", "YearBuilt"]]

y_train = train_and_val["ClosePrice"]

In [147]:
encoder = TargetEncoder(smooth="auto", cv=5, random_state=42)

x_train_and_val_encoded = encoder.fit_transform(to_encode_train, y_train)
x_test_encoded = encoder.transform(to_encode_test)

x_train_and_val_encoded_df = pd.DataFrame(x_train_and_val_encoded, columns=to_encode_train.columns)
x_test_encoded_df = pd.DataFrame(x_test_encoded, columns=to_encode_train.columns)

In [148]:
new_train_val = train_and_val.join(x_train_and_val_encoded_df, rsuffix="_no_log_enc")

new_training = new_train_val[~(new_train_val["PropertyMonth"] == "2026-03-01")]
new_validation = new_train_val[new_train_val["PropertyMonth"] == "2026-03-01"]

In [149]:
new_validation.columns

Index(['ViewYN', 'PoolPrivateYN', 'ListingKey', 'ClosePrice', 'Latitude',
       'Longitude', 'LivingArea', 'ListOfficeName', 'CountyOrParish',
       'AttachedGarageYN', 'ParkingTotal', 'YearBuilt', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'FireplaceYN',
       'Stories', 'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'PropertyMonth', 'ClosePriceLog', 'LivingAreaLog',
       'LotSizeSquareFeetLog', 'AssociationFeeLog', 'Bamboo', 'Brick',
       'Carpet', 'Concrete', 'Laminate', 'SeeRemarks', 'Stone', 'Tile',
       'Unknown', 'Vinyl', 'Wood', 'ListOfficeName_enc', 'CountyOrParish_enc',
       'HighSchoolDistrict_enc', 'City_enc', 'PostalCode_enc', 'YearBuilt_enc',
       'ListOfficeName_no_log_enc', 'CountyOrParish_no_log_enc',
       'HighSchoolDistrict_no_log_enc', 'City_no_log_enc',
       'PostalCode_no_log_enc', 'YearBuilt_no_log_enc'],
      dtype='obj

In [150]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_no_log_enc", "CountyOrParish_no_log_enc", "HighSchoolDistrict_no_log_enc",
            "City_no_log_enc", "PostalCode_no_log_enc", "YearBuilt_no_log_enc"]

In [151]:
x_train = new_training[common_cols]
y_train = new_training["ClosePrice"]

x_val = new_validation[common_cols]  
y_val = new_validation["ClosePrice"]

x_test = testing.join(x_test_encoded_df, rsuffix="_no_log_enc")
x_test = x_test[common_cols]  
y_test = testing["ClosePrice"]

In [152]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

# Note: put False in the metrics function to not perform the np.exp() which usually undos the log transform
r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train, False)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val, False)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test, False)

In [153]:
model_two = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_two

,r2,MAPE,MdAPE,MAE
Train Set,0.812124,18.225638,14.131572,$180267.78
Validation Set,0.803517,18.477659,14.449222,$185489.79
Test Set,0.826571,17.611307,13.615927,$180845.1


When ClosePrice is used as the target variable instead of ClosePriceLog, the model drops in performance slightly. This is a good sign, as it suggests that the log transform of ClosePrice is justified. Additionally, the difference between the two models' performance is not big enough to be suspicious that the log transform is misrepresenting that data.

## Model Three: BedBathRatio and PropertyAge

Two features, BedBathRatio and PropertyAge, were created during the feature engineering done in week six. Test how model performance changes with the inclusion of these new features. Additionally, go back to using ClosePriceLog as the target variable for the remaining models.

In [154]:
# Read in the training and test sets created in week six

train_and_val = pd.read_csv("feat_eng_train.csv")
testing = pd.read_csv("feat_eng_test.csv")

In [155]:
training = train_and_val[~(train_and_val["PropertyMonth"] == "2026-03-01")]
validation = train_and_val[train_and_val["PropertyMonth"] == "2026-03-01"]

print(train_and_val.shape, training.shape, validation.shape)

(265296, 153) (255004, 153) (10292, 153)


In [156]:
common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "HighSchoolDistrict_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge"]

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [157]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [158]:
model_three = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_three

,r2,MAPE,MdAPE,MAE
Train Set,0.836164,13.531113,10.428432,$152299.87
Validation Set,0.825305,13.927309,10.922089,$158684.91
Test Set,0.846471,13.150412,10.302092,$154892.97


Model performance does increase very slightly, but not to a notable decree. Clearly, more feature engineering is necessary to see real improvement.

## Model Four: Improved School Districts

During week six, the existing dataset was joined with the California School District Areas 2024-2025 dataset (https://data.ca.gov/@california-department-of-education/california-school-district-areas-2024-25) to improve the locational data. Test how the model performance changes with these new features.

### Both Counts and Percents

In [159]:
# The new dataset contains school district names, as well as demographic data from that district
# This section will include both the percent and count columns that describe demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AApct_high', 'AAcount_high', 'AIpct_high',
            'AIcount_high', 'ASpct_high', 'AScount_high', 'FIpct_high', 'FIcount_high', 'HIpct_high', 'HIcount_high', 'PIpct_high',
            'PIcount_high', 'WHpct_high', 'WHcount_high', 'MRpct_high', 'MRcount_high', 'NRpct_high', 'NRcount_high', 'ELpct_high',
            'ELcount_high', 'FOSpct_high', 'FOScount_high','HOMpct_high','HOMcount_high','MIGpct_high','MIGcount_high','SWDpct_high',
            'SWDcount_high','SEDpct_high','SEDcount_high', 'EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AApct_elem',
            'AAcount_elem','AIpct_elem','AIcount_elem','ASpct_elem','AScount_elem','FIpct_elem','FIcount_elem','HIpct_elem','HIcount_elem',
            'PIpct_elem','PIcount_elem','WHpct_elem','WHcount_elem','MRpct_elem','MRcount_elem','NRpct_elem','NRcount_elem','ELpct_elem',
            'ELcount_elem','FOSpct_elem','FOScount_elem','HOMpct_elem','HOMcount_elem','MIGpct_elem','MIGcount_elem','SWDpct_elem',
            'SWDcount_elem','SEDpct_elem','SEDcount_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AApct_unif',
            'AAcount_unif','AIpct_unif','AIcount_unif','ASpct_unif','AScount_unif','FIpct_unif','FIcount_unif','HIpct_unif','HIcount_unif',
            'PIpct_unif','PIcount_unif','WHpct_unif','WHcount_unif','MRpct_unif','MRcount_unif','NRpct_unif','NRcount_unif','ELpct_unif',
            'ELcount_unif','FOSpct_unif','FOScount_unif','HOMpct_unif','HOMcount_unif','MIGpct_unif','MIGcount_unif','SWDpct_unif',
            'SWDcount_unif','SEDpct_unif','SEDcount_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [160]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [161]:
model_four_v1 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v1

,r2,MAPE,MdAPE,MAE
Train Set,0.844110,12.918196,9.780404,$146197.42
Validation Set,0.832815,13.315817,10.189271,$152831.88
Test Set,0.855653,12.550245,9.552526,$148375.8


Model performance does increase a noticeable amount with the addition of these new features. This makes sense, since a desirable school district can be a major reason why people buy certain residential properties.

### Only Percents

The previous model used both counts and percentages of demographic data, however this is a lot of features to be using and might be making our model overly complex. Let's test whether model performance changes when we use only counts or only percents. In this section, use only percents for demographic data.

In [162]:
# Test if model performance changes when only the percent columns are used for demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AApct_high', 'AIpct_high',
            'ASpct_high','FIpct_high','HIpct_high','PIpct_high',
            'WHpct_high','MRpct_high', 'NRpct_high', 'ELpct_high',
            'FOSpct_high', 'HOMpct_high','MIGpct_high','SWDpct_high',
            'SEDpct_high','EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AApct_elem',
            'AIpct_elem','ASpct_elem','FIpct_elem','HIpct_elem',
            'PIpct_elem','WHpct_elem','MRpct_elem','NRpct_elem','ELpct_elem',
            'FOSpct_elem','HOMpct_elem','MIGpct_elem','SWDpct_elem',
            'SEDpct_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AApct_unif',
            'AIpct_unif','ASpct_unif','FIpct_unif','HIpct_unif',
            'PIpct_unif','WHpct_unif','MRpct_unif','NRpct_unif','ELpct_unif',
            'FOSpct_unif','HOMpct_unif','MIGpct_unif','SWDpct_unif',
            'SEDpct_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [163]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [164]:
model_four_v2 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v2

,r2,MAPE,MdAPE,MAE
Train Set,0.842274,13.081806,9.966240,$147720.61
Validation Set,0.830803,13.473392,10.495274,$154274.48
Test Set,0.854148,12.695159,9.767038,$149713.85


Model performance decreased a very slight amount when using only percents, rather than both percents and counts. This makes some sense, since the inclusion of demographic counts subtly encodes the population of a school district. An area that serves more poeple may be indicative of more a desirable school district, so the counts data may be essential to encoding desirablilty (which should correlate with ClosePrice).

### Only Counts

In this section, use only counts for demographic data.

In [165]:
# Test if model performance changes when only the count columns are used for demographic information

common_cols = ["ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
            "Longitude", "Latitude", 
            "LivingAreaLog", "AssociationFeeLog", "LotSizeSquareFeetLog" ,
            "ParkingTotal", "GarageSpaces", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "Stories",
            "Bamboo", "Brick", "Carpet", "Concrete", "Laminate", "SeeRemarks", "Stone", "Tile", "Unknown", "Vinyl", "Wood",
            "ListOfficeName_enc", "CountyOrParish_enc", "City_enc", "PostalCode_enc", "YearBuilt_enc",
            "BedBathRatio", "PropertyAge",
            "DistrictName_elem_enc", "DistrictName_unif_enc", "DistrictName_high_enc",
            'EnrollTotal_high', 'EnrollCharter_high', 'EnrollNonCharter_high', 'AAcount_high', 'AIcount_high',
            'AScount_high','FIcount_high','HIcount_high','PIcount_high',
            'WHcount_high','MRcount_high', 'NRcount_high', 'ELcount_high',
            'FOScount_high', 'HOMcount_high','MIGcount_high','SWDcount_high',
            'SEDcount_high','EnrollTotal_elem','EnrollCharter_elem','EnrollNonCharter_elem','AAcount_elem',
            'AIcount_elem','AScount_elem','FIcount_elem','HIcount_elem',
            'PIcount_elem','WHcount_elem','MRcount_elem','NRcount_elem','ELcount_elem',
            'FOScount_elem','HOMcount_elem','MIGcount_elem','SWDcount_elem',
            'SEDcount_elem','EnrollTotal_unif','EnrollCharter_unif','EnrollNonCharter_unif','AAcount_unif',
            'AIcount_unif','AScount_unif','FIcount_unif','HIcount_unif',
            'PIcount_unif','WHcount_unif','MRcount_unif','NRcount_unif','ELcount_unif',
            'FOScount_unif','HOMcount_unif','MIGcount_unif','SWDcount_unif',
            'SEDcount_unif']

x_train = training[common_cols]
y_train = training["ClosePriceLog"]

x_val = validation[common_cols]  
y_val = validation["ClosePriceLog"]

x_test = testing[common_cols]
y_test = testing["ClosePriceLog"]

In [166]:
model = LinearRegression()
model.fit(x_train, y_train)

pred_train = model.predict(x_train)
pred_val = model.predict(x_val)
pred_test = model.predict(x_test)

r2_train, mape_train, mdape_train, mae_train = metrics(y_train, pred_train)
r2_val, mape_val, mdape_val, mae_val = metrics(y_val, pred_val)
r2_test, mape_test, mdape_test, mae_test = metrics(y_test, pred_test)

In [167]:
model_four_v3 = metric_df(r2_train, mape_train, mdape_train, mae_train, 
                        r2_val, mape_val, mdape_val, mae_val,
                        r2_test, mape_test, mdape_test, mae_test)
model_four_v3

,r2,MAPE,MdAPE,MAE
Train Set,0.841819,13.072119,9.949275,$147861.44
Validation Set,0.831405,13.415999,10.291343,$153929.4
Test Set,0.853570,12.669389,9.652377,$149848.16


When using only counts for demographic data, model performance continued to decrease, but only slightly. The model performs better when both counts and percents are used for demographic data, but the difference is so small that using only counts or only percentages is a valid way to reduce complexity without losing predictive power.

# Summary

This table presents the best performing model from each section.

| Model | Test r2 | Test MAPE | Test MdAPE | Test MAE | 
| ----- | ------- | --------- | ---------- | -------- |
| Model One (Baseline) | 0.845539 | 13.254369 | 10.399052 | \$155,596 |
| Model Two (Removing Log Transform on Target Variable) | 0.826571 | 17.611307 | 13.615927 | \$155,840 |
| Model Three (Feature Engineering Additions) | 0.846471 | 13.150412 | 10.302092 | \$154,892 |
| **Model Four (School District Information)** | **0.855653** | **12.550245** | **9.552526** | **\$148,375**|

These are very good metrics for a simple linear regression model, and they suggest that the ClosePrice has a largely linear relationship with many of the features. However, linear regression models are limited in that they have difficulty capturing non-linear relationships. Thus, we should test the dataset further with non-linear models (such as decision trees or random forests) in order to better predict ClosePrice when there exist non-linear relationships in the data.